In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("WELFake_Dataset.csv")

# Drop first column (ID column with empty name)
df = df.iloc[:, 1:]

# Drop missing values
df = df.dropna()

# Combine title and text
df["content"] = df["title"] + " " + df["text"]

X = df["content"]
y = df["label"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train logistic regression model
model = LogisticRegression(
    max_iter=1000,
    solver="liblinear"
)

model.fit(X_train_tfidf, y_train)

# Evaluate model
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.949119373776908
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      7006
           1       0.95      0.96      0.95      7302

    accuracy                           0.95     14308
   macro avg       0.95      0.95      0.95     14308
weighted avg       0.95      0.95      0.95     14308



In [3]:
# Evaluate on training set
y_train_pred = model.predict(X_train_tfidf)

print("TRAINING SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

print("\nTEST SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


TRAINING SET PERFORMANCE
Accuracy: 0.9707141484212549
              precision    recall  f1-score   support

           0       0.98      0.96      0.97     28022
           1       0.96      0.98      0.97     29207

    accuracy                           0.97     57229
   macro avg       0.97      0.97      0.97     57229
weighted avg       0.97      0.97      0.97     57229


TEST SET PERFORMANCE
Accuracy: 0.949119373776908
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      7006
           1       0.95      0.96      0.95      7302

    accuracy                           0.95     14308
   macro avg       0.95      0.95      0.95     14308
weighted avg       0.95      0.95      0.95     14308



In [4]:
from sklearn.metrics import confusion_matrix

print("TRAIN confusion matrix:")
print(confusion_matrix(y_train, y_train_pred))

print("\nTEST confusion matrix:")
print(confusion_matrix(y_test, y_pred))


TRAIN confusion matrix:
[[26968  1054]
 [  622 28585]]

TEST confusion matrix:
[[6605  401]
 [ 327 6975]]


In [5]:
def test_article(title, text, model, vectorizer, crisis=False):
    # Combine title and text
    content = title + " " + text

    # Vectorize input
    content_tfidf = vectorizer.transform([content])

    # Get probabilities
    prob_fake, prob_real = model.predict_proba(content_tfidf)[0]

    # Apply threshold
    threshold = 0.8 if crisis else 0.5

    classification = (
        "Most likely real" if prob_real >= threshold else "Most likely fake"
    )

    return classification, prob_real, prob_fake


In [6]:
title = "Scientists Confirm Drinking Coffee Makes People Live to 150 Years Old"
text = "A group of scientists reportedly discovered that drinking three cups of coffee a day can extend human life to over 150 years. The study, which has not yet been published or reviewed, claims participants showed “near-immortal cellular behavior.”"

result, prob_real, prob_fake = test_article(
    title=title,
    text=text,
    model=model,
    vectorizer=vectorizer,
    crisis=True  # set False for normal mode
)

print("Classification:", result)
print("Probability real:", round(prob_real, 3))
print("Probability fake:", round(prob_fake, 3))


Classification: Most likely fake
Probability real: 0.749
Probability fake: 0.251


How to improve

Here are incremental, defensible improvements:

1. Penalize “extraordinary claims”

Add features like:

Presence of extreme numbers (“150 years”)

Superlatives (“immortal”, “guaranteed”)

2. Source credibility (huge)

If no known source or named institution → penalty.

3. Crisis-aware stricter thresholds (you already do this)

This is not a hack — it’s policy-aware ML.

4. Confidence banding (very clean idea)
> 0.85 → Most likely real
0.65–0.85 → Uncertain (human review)
< 0.65 → Most likely fake


This would’ve flagged your coffee article as uncertain, which is correct.